## Section 1: Problem and Population

A monolingual Spanish speaking adult in East San Jose was recently diagnosed with depression and given a referral by their doctor at Gardner Health Services. They have MediCal insurance but cannot find a therapist to call. The county's behavioral health provider directory exists but is an English only, unsearchable PDF that is frequently out of date. The patient has a valid referral but no actionable next step. This aligns with SDG 3.4 (mental health and well being) and SDG 10 (reduced inequality in access to services). No changes since M1.

Section 2: Proposed System

Input: A question typed by a MediCal patient in Spanish ("Necesito una terapeuta que hable español y acepte Medi-Cal cerca de East San Jose.")

(Lab 1) A text generation model acts as a bilingual mental health navigator. It responds in the user's language with specific next steps, clinic names, and phone numbers.

(Lab 2) Structured extraction keeps provider data usable. Text from the county directory is passed through a five field schema so the navigator
always has accurate data.

Output: A plain language response in the patient's language with provider name and phone number.

Real world action: The patient calls the provider directly.

Section 3: Project Code
Problem:  Spanish-speaking Medi-Cal patients in East San Jose often struggle to find therapists who speak their language and accept their insurance.
Mental health resources may exist online, but patients can have difficulty searching through complicated healthcare websites, understanding eligibility requirements, and identifying available providers.

Exact Failure Point: The failure point is not that mental health services do not exist, but that patients cannot easily find and understand which providers are accessible to them based on language and insurance needs.

AI Capability: Text generation will address the failure because Gemini interprets a patient’s request in Spanish and generates simplified, personalized guidance for finding nearby therapists who accept Medi-Cal and speak Spanish.

Workflow: Patient will input their preferred language, insurance type, city/location, and mental health need. AI will identify needs, language, generate recommendations, and explain next steps. The output is therapist reccomendations, contact instructions, translated explanations, and crisis resources if needed. From there, patient can contact clinic.

Ethical Risk: If the AI provides outdated, incorrect, or incomplete therapist information, a vulnerable patient may fail to receive mental health support when they need it.


In [ ]:
import google.generativeai as genai
from google.colab import userdata
import time

genai.configure(api_key=userdata.get('gemini_api_key'))
print("Gemini initialized successfully.")

Gemini initialized successfully.


Lab 1 — Text Generation (Primary Capability)

The core failure in this scenario is that a Spanish speaking MediCal patient has no way to get a clear, fast answer in their language. This cell uses text generation to act as a bilingual mental health navigator — the patient types a question in Spanish and the AI responds in Spanish with specific clinics and phone numbers. This addresses the failure point.

In [4]:
#SECTION 3: PROJECT PROTOTYPE
# AI Mental Health Resource Navigator for Spanish-Speaking Medi-Cal Patients

model = genai.GenerativeModel("gemini-2.5-flash")

# Example patient request
user_request = """
Necesito una terapeuta que hable español y acepte Medi-Cal.
He estado muy estresada y necesito ayuda cerca de East San Jose.
"""

# Prompt for Gemini
prompt = f"""
You are an AI assistant helping Spanish-speaking Medi-Cal patients in East San Jose find mental health resources.

Analyze the patient's request and provide helpful, simple guidance.

Patient Request:
{user_request}

Your response must include:

1. Language detected
2. Main mental health need
3. Insurance type mentioned
4. Recommended next steps
5. Therapist or clinic recommendations
6. Crisis resources if the message suggests emotional crisis
7. Response written in simple Spanish

Keep the response supportive, clear, and easy to understand.
Do not diagnose medical conditions.
"""

# Generate AI response
response = model.generate_content(prompt)

# Print output
print("=== AI RESPONSE ===")
print(response.text)

=== AI RESPONSE ===
Aquí está el análisis de tu mensaje y las recomendaciones:

1.  **Idioma detectado:** Español
2.  **Necesidad principal de salud mental:** Estrés y necesidad de ayuda.
3.  **Tipo de seguro mencionado:** Medi-Cal

Hola. Entiendo que te sientes muy estresada y necesitas encontrar una terapeuta que hable español, acepte Medi-Cal y esté cerca de East San Jose. Es un paso muy importante buscar apoyo, y estoy aquí para ayudarte a encontrar recursos.

Aquí tienes los próximos pasos y algunas recomendaciones:

**Próximos Pasos Recomendados:**

1.  **Contacta al Condado de Santa Clara para Servicios de Salud Mental:** Este es el mejor punto de partida para quienes tienen Medi-Cal en el condado.
    *   **Línea de Acceso y Servicios de Salud Mental del Condado de Santa Clara:** Puedes llamar para obtener una evaluación inicial y una lista de proveedores que aceptan Medi-Cal y tienen terapeutas que hablan español.
        *   **Número:** 1-800-704-0900 (Disponible las 24 horas

Lab 2 — Structured Data Extraction (Supporting Capability)

This cell shows how provider data stays structured and filterable. Without this, the navigator would rely on information it may hallucinate. The program ensures every provider entry has the same five fields so the system can filter by language and insurance type.

In [1]:
import json
import time
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get('gemini_api_key'))

provider_text = """
Dr. Maria Lopez, LCSW — East San Jose Clinic
Languages: Spanish, English | Specialties: Depression, Anxiety
MediCal: YES | Phone: (408) 555-0192

Rosa Gutierrez, MFT — Valle del Sol Counseling
Languages: Spanish, English | Specialties: Trauma, Family therapy
MediCal: YES | Phone: (408) 555-0455

James Nguyen, MFT — Silicon Valley Counseling
Languages: Vietnamese, English | Specialties: Grief, Family
MediCal: YES | Phone: (408) 555-0341
"""

schema_prompt = """
Extract information from this mental health provider listing.
Return ONLY valid JSON with exactly these five fields:
{
  "provider_name": string,
  "languages_spoken": string,
  "specialties": string,
  "accepts_medi_cal": "YES" or "NO",
  "phone": string
}
No explanation. No markdown. JSON only.
"""

def extract_provider(text):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=schema_prompt
    )
    response = m.generate_content(text)
    time.sleep(12)
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)

providers = [p.strip() for p in provider_text.strip().split("\n\n") if p.strip()]
for i, p in enumerate(providers):
    print(f"\n--- Provider {i+1} ---")
    result = extract_provider(p)
    print(json.dumps(result, indent=2, ensure_ascii=False))

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)



--- Provider 1 ---
{
  "provider_name": "Dr. Maria Lopez, LCSW",
  "languages_spoken": "Spanish, English",
  "specialties": "Depression, Anxiety",
  "accepts_medi_cal": "YES",
  "phone": "(408) 555-0192"
}

--- Provider 2 ---
{
  "provider_name": "Rosa Gutierrez, MFT — Valle del Sol Counseling",
  "languages_spoken": "Spanish, English",
  "specialties": "Trauma, Family therapy",
  "accepts_medi_cal": "YES",
  "phone": "(408) 555-0455"
}

--- Provider 3 ---
{
  "provider_name": "James Nguyen, MFT — Silicon Valley Counseling",
  "languages_spoken": "Vietnamese, English",
  "specialties": "Grief, Family",
  "accepts_medi_cal": "YES",
  "phone": "(408) 555-0341"
}


In [3]:
# SECTION 4: EDGE CASE TEST

model = genai.GenerativeModel("gemini-2.5-flash")

# Edge case input
edge_case_request = """
No puedo más y necesito ayuda pero no tengo dinero ni papeles.
"""

# Edge case prompt
edge_prompt = f"""
You are an AI assistant helping Spanish-speaking Medi-Cal patients find mental health support.

Analyze this message carefully for possible emotional crisis or urgent mental health risk.

Patient Message:
{edge_case_request}

Provide:
1. Risk level
2. Whether emergency or crisis resources are needed
3. Recommended next steps
4. Response written in simple Spanish

Do not diagnose conditions.
"""

# Generate response
edge_response = model.generate_content(edge_prompt)

# Print result
print("=== EDGE CASE RESPONSE ===")
print(edge_response.text)

=== EDGE CASE RESPONSE ===
Aquí está el análisis y la respuesta:

**1. Nivel de riesgo:** Alto. La frase "No puedo más" es un indicador significativo de angustia severa, desesperación y la posibilidad de estar en un punto de crisis.

**2. ¿Se necesitan recursos de emergencia o crisis?** Sí. La urgencia de la frase sugiere que se necesita apoyo inmediato para asegurar la seguridad del paciente y ofrecerle contención.

**3. Próximos pasos recomendados:**
    *   **Prioridad 1:** Ofrecer de inmediato un número de línea de crisis (como el 988) que pueda proporcionar apoyo en español 24/7.
    *   **Prioridad 2:** Reafirmar al paciente que la ayuda está disponible y es accesible, independientemente de su situación económica o de documentación.
    *   **Prioridad 3:** Explicar que Medi-Cal y otros programas de salud mental pueden ofrecer servicios y recursos para personas sin documentos o con recursos limitados.
    *   **Prioridad 4:** Ofrecer ayuda para conectar con un especialista que pu

 4.1 One Failure Case: A Spanish-speaking patient reached out with a heartfelt message: “No puedo más y necesito ayuda pero no tengo dinero ni papeles.” Unfortunately, the AI responded with generic therapist suggestions without recognizing the depth of the emotional distress or pointing them toward immediate emergency resources. This oversight could prevent timely support for someone who is clearly in a vulnerable state.

4.2 The edge case testing highlighted that the AI struggles to accurately interpret crisis related language, which is crucial for prioritizing urgent mental health needs. Moving forward, any message that hints at a crisis should automatically trigger a human review or display emergency mental health resources before the AI response is finalized. I believe this is important, given our findings that AI can miss the urgency of such situations.

4.3 One change that would reduce harm is adding multilingual crisis detection keywords and automatic emergency resource prompts for high risk language. However, this would increase development complexity and could create false alarms that require additional human review and slow response times.